# Prop-DeOccNet Training — Kaggle

**Sebelum mulai:**
1. Aktifkan GPU: *Settings → Accelerator → GPU T4 x2* (atau P100)
2. Aktifkan Internet: *Settings → Internet → On*
3. Tambahkan dataset Roboflow COCO: *+ Add Input → cari dataset kamu*

---

## Step 1 — Cek GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        vram = round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1)
        print(f'  GPU {i}: {name} ({vram} GB)')
else:
    print('WARNING: GPU tidak aktif — aktifkan di Settings → Accelerator')

## Step 2 — Konfigurasi

Cek path dataset di panel kiri → `/kaggle/input/<DATASET_NAME>/`

In [ ]:
# ── Edit sesuai setup kamu ────────────────────────────────────────────────────
DATASET_NAME            = "daun-kelengkeng-itoh-1000"  # <-- ganti
REPO_URL                = ""   # GitHub URL, atau kosong jika pakai dataset kode
CODE_DATASET_NAME       = "labeling-daun-itoh-code"    # <-- ganti jika REPO_URL kosong
CHECKPOINT_DATASET_NAME = ""   # dataset checkpoint dari run sebelumnya, kosong = fresh

EPOCHS      = 50
BATCH_SIZE  = 4     # T4 16GB: 4 | P100 16GB: 6 | jika OOM turunkan ke 2
IMAGE_SIZE  = 512
BACKBONE    = "resnet101"

# ── Path (set DATASET_PATH langsung jika path mount tidak standar) ────────────
DATASET_PATH   = f"/kaggle/input/{DATASET_NAME}"
WORK_DIR       = "/kaggle/working/labeling-daun-itoh"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
RUNS_DIR       = "/kaggle/working/runs"
CONFIG_PATH    = f"{WORK_DIR}/training/config_kaggle.yaml"

print(f"Dataset : {DATASET_PATH}")
print(f"Work    : {WORK_DIR}")
print(f"Ckpt    : {CHECKPOINT_DIR}")

## Step 3 — Setup Kode Project

In [ ]:
import os, sys, shutil
from pathlib import Path

if REPO_URL:
    if not Path(WORK_DIR).exists():
        if os.system(f"git clone {REPO_URL} {WORK_DIR}") != 0:
            raise RuntimeError("git clone gagal — pastikan Internet aktif dan URL benar")
    print(f"Kode di-clone ke: {WORK_DIR}")
else:
    CODE_SRC = f"/kaggle/input/{CODE_DATASET_NAME}"
    if not Path(CODE_SRC).exists():
        raise FileNotFoundError(f"Dataset kode tidak ditemukan: {CODE_SRC}")
    if not Path(WORK_DIR).exists():
        shutil.copytree(CODE_SRC, WORK_DIR)
    print(f"Kode di-copy dari: {CODE_SRC}")

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print("Working dir:", os.getcwd())
print("Isi folder :", sorted(os.listdir('.')))

## Step 4 — Install Dependencies

> Jika muncul error NumPy setelah install: ***Session → Restart & Run All***.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "cython>=3.0.0", "-q"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "pycocotools", "--no-binary", "pycocotools", "--no-cache-dir", "-q"],
    check=True
)
import numpy as np, torch, pycocotools._mask, albumentations as A
print(f"numpy={np.__version__}  torch={torch.__version__}  CUDA={torch.cuda.is_available()}")
print(f"albumentations={A.__version__}  pycocotools=OK")

## Step 5 — Deteksi Dataset & Tulis Config

Auto-detect `_annotations.coco.json` di mana pun letaknya (menangani struktur nested).

In [ ]:
import yaml, json
from pathlib import Path

# ── 1. Tampilkan struktur folder ──────────────────────────────────────────────
dataset_root = Path(DATASET_PATH)
print(f"Struktur: {dataset_root}\n" + "-"*60)
for item in sorted(dataset_root.rglob("*")):
    depth = len(item.relative_to(dataset_root).parts)
    if depth > 4:
        continue
    indent = "  " * (depth - 1)
    tag = "/" if item.is_dir() else f"  ({item.stat().st_size//1024}KB)"
    print(f"{indent}{item.name}{tag}")
print("-"*60)

# ── 2. Auto-detect annotation files ──────────────────────────────────────────
SPLIT_ALIASES = {
    "train": "train", "valid": "valid",
    "val":   "valid", "test":  "test", "evaluation": "test",
}
ann_files = sorted(dataset_root.rglob("_annotations.coco.json"))
if not ann_files:
    raise FileNotFoundError(
        f"Tidak ada '_annotations.coco.json' di {dataset_root}\n"
        "Pastikan dataset sudah di-add via '+ Add Input'."
    )
SPLIT_PATHS = {}
for ann in ann_files:
    canonical = SPLIT_ALIASES.get(ann.parent.name.lower())
    if canonical:
        SPLIT_PATHS[canonical] = str(ann)
if not SPLIT_PATHS:
    raise ValueError(
        f"Nama folder tidak dikenali: {[f.parent.name for f in ann_files]}\n"
        f"Didukung: {list(SPLIT_ALIASES.keys())}"
    )

print("\nPath terdeteksi:")
for split, path in sorted(SPLIT_PATHS.items()):
    print(f"  {split:5s} → {path}")

# ── 3. Verifikasi ─────────────────────────────────────────────────────────────
print("\nVerifikasi dataset:")
for split in ["train", "valid", "test"]:
    path = SPLIT_PATHS.get(split)
    if not path:
        print(f"  [{split:5s}] tidak ditemukan")
        continue
    with open(path) as f:
        data = json.load(f)
    img_dir = Path(path).parent
    n_files = len(list(img_dir.glob("*.jpg"))) + len(list(img_dir.glob("*.png")))
    print(f"  [{split:5s}] {len(data['images']):4d} gambar | "
          f"{len(data['annotations']):5d} anotasi | {n_files} file gambar")
print("\nKategori:")
with open(SPLIT_PATHS["train"]) as f:
    for c in json.load(f)["categories"]:
        print(f"  id={c['id']} → {c['name']}")

# ── 4. Tulis config ───────────────────────────────────────────────────────────
import torch
_n_cpu = 4 if torch.cuda.device_count() > 1 else 2  # T4x2=4core, P100/T4=2core
config = {
    "train_json": SPLIT_PATHS.get("train"),
    "val_json":   SPLIT_PATHS.get("valid"),
    "test_json":  SPLIT_PATHS.get("test"),
    "images_dir": None, "num_classes": 3,
    "backbone": BACKBONE, "pretrained_backbone": True,
    "aspp_rates": [6, 12, 18, 24], "aspp_out_channels": 256,
    "trainable_backbone_layers": 3, "use_boundary_head": True,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE,
    "num_workers": _n_cpu, "image_size": IMAGE_SIZE, "mosaic_prob": 0.3,
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std":  [0.229, 0.224, 0.225],
    "optimizer": "adam", "lr": 1e-4, "weight_decay": 1e-4,
    "lr_scheduler": "cosine", "lr_min": 1e-6,
    "loss_weights": {"focal": 1.0, "dice": 1.0, "boundary": 1.0},
    "checkpoint_dir": CHECKPOINT_DIR, "save_every": 5, "resume": None,
    "tensorboard_dir": RUNS_DIR, "log_every": 20,
    "occlusion_thresholds": None,   # None = auto dari persentil training
    "occlusion_filter": None,
    "occlusion_stats": True,
}
Path(CONFIG_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(CONFIG_PATH, "w") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)
print(f"\nConfig ditulis ke: {CONFIG_PATH}")
print(f"num_workers otomatis: {_n_cpu} ({'T4 x2' if _n_cpu == 4 else 'P100 / T4 single'})")

## Step 6 — Statistik Oklusi

**Metode: bbox overlap ratio** (bukan mask overlap).
Roboflow menganotasi hanya bagian *terlihat* dari setiap daun → mask tidak pernah overlap.
Bounding box mencakup seluruh extent daun, sehingga bbox dua daun yang saling menutupi AKAN overlap.

```
occlusion_ratio = (Σ luas bbox − piksel unik) / Σ luas bbox
```

**Threshold otomatis** dari persentil ke-33 & ke-67 distribusi training → distribusi ~seimbang.

| Level | Kondisi |
|-------|--------|
| **Rendah** | ratio < p33 (1/3 terbawah) |
| **Sedang** | p33 ≤ ratio < p67 (1/3 tengah) |
| **Tinggi** | ratio ≥ p67 (1/3 teratas) |

In [ ]:
from training.dataset import compute_dataset_occlusion_stats
import numpy as np

# ── Hitung train dulu (auto-threshold dari distribusinya) ─────────────────────
print("Oklusi [train]:")
train_stats = compute_dataset_occlusion_stats(
    SPLIT_PATHS["train"],
    thresholds=None,  # auto persentil ke-33 & ke-67
    verbose=True
)
CALIBRATED_THRESHOLDS = train_stats["thresholds"]  # simpan untuk val/test

# ── Terapkan threshold yang sama ke val/test ──────────────────────────────────
all_stats = {"train": train_stats}
for split in ["valid", "test"]:
    ann_path = SPLIT_PATHS.get(split)
    if not ann_path:
        continue
    print(f"\nOklusi [{split}]:")
    all_stats[split] = compute_dataset_occlusion_stats(
        ann_path, thresholds=CALIBRATED_THRESHOLDS, verbose=True
    )

# ── Tabel ringkasan ────────────────────────────────────────────────────────────
print("\n" + "="*62)
print(f"{'Split':<8} {'Rendah':>12} {'Sedang':>12} {'Tinggi':>12} {'Total':>6}")
print("-"*62)
for split, s in all_stats.items():
    t = max(s["total"], 1)
    r = f"{s['rendah']} ({s['rendah']/t:.0%})"
    m = f"{s['sedang']} ({s['sedang']/t:.0%})"
    h = f"{s['tinggi']} ({s['tinggi']/t:.0%})"
    print(f"{split:<8} {r:>12} {m:>12} {h:>12} {s['total']:>6}")
print("="*62)
low_t, high_t = CALIBRATED_THRESHOLDS
print(f"Threshold dikalibrasi: rendah<{low_t:.4f}, sedang={low_t:.4f}–{high_t:.4f}, tinggi≥{high_t:.4f}")

## Step 7 — Load Checkpoint (Opsional)

In [ ]:
import shutil
from pathlib import Path

Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
if CHECKPOINT_DATASET_NAME:
    ckpt_src = Path(f"/kaggle/input/{CHECKPOINT_DATASET_NAME}")
    if not ckpt_src.exists():
        raise FileNotFoundError(f"Dataset checkpoint tidak ditemukan: {ckpt_src}")
    pth_files = list(ckpt_src.glob("*.pth"))
    for f in pth_files:
        shutil.copy2(f, Path(CHECKPOINT_DIR) / f.name)
    print(f"Loaded {len(pth_files)} checkpoint(s):")
    for f in sorted(Path(CHECKPOINT_DIR).glob("*.pth")):
        print(f"  {f.name}  ({f.stat().st_size/1e6:.0f} MB)")
else:
    print("Training dimulai dari awal.")

## Step 8 — Training

| GPU | batch_size | Estimasi 50 epoch |
|-----|-----------|-------------------|
| T4 x2 (1 GPU aktif) | 4 | ~5–8 jam |
| P100 16GB | 4–6 | ~4–6 jam |

> **P100 vs T4:** P100 tidak punya Tensor Cores, jadi AMP (FP16) hemat memori tapi speedup-nya lebih kecil dari T4. Namun memory bandwidth P100 lebih tinggi (~732 GB/s vs 300 GB/s), sehingga per-epoch bisa lebih cepat.
> RAM P100 di Kaggle hanya ~13GB (vs T4 x2 ~29GB) — jangan tambah mosaic_prob terlalu tinggi.

In [ ]:
import sys
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

from training.train import train
train(config_path=CONFIG_PATH)

## Step 9 — TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/runs/

## Step 10 — Evaluasi Model Terbaik

In [ ]:
import torch, yaml
from pathlib import Path
from torch.utils.data import DataLoader
from training.model import PropDeOccNet
from training.dataset import DaunDataset, collate_fn, get_val_transforms
from training.evaluate import evaluate

cfg = yaml.safe_load(open(CONFIG_PATH))
best_ckpt = Path(CHECKPOINT_DIR) / "best.pth"
if not best_ckpt.exists():
    raise FileNotFoundError(f"best.pth tidak ada — training belum selesai?")

model = PropDeOccNet(
    num_classes=cfg["num_classes"], backbone=cfg["backbone"],
    aspp_rates=cfg["aspp_rates"], use_boundary_head=cfg["use_boundary_head"],
)
ckpt = torch.load(best_ckpt, map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])
model = model.to("cuda")
print(f"Loaded epoch {ckpt['epoch']+1} (BF={ckpt.get('bf_score',0):.4f})")

test_ds = DaunDataset(
    coco_json_path=cfg["test_json"],
    images_dir=cfg.get("images_dir"),
    transforms=get_val_transforms(cfg["image_size"]),
)
test_loader = DataLoader(test_ds, batch_size=1, collate_fn=collate_fn, num_workers=2)
metrics = evaluate(model, test_loader, device="cuda")
print("\n=== Test Set Results ===")
for k, v in metrics.items():
    print(f"  {k:<18}: {v:.4f}")

## Step 11 — Output & Download

Download checkpoint via panel kanan → *Output* tab.
Untuk melanjutkan di sesi berikutnya: upload `checkpoints/` sebagai Kaggle dataset baru → set `CHECKPOINT_DATASET_NAME`.

In [ ]:
from pathlib import Path
working = Path("/kaggle/working")
ckpt_dir = working / "checkpoints"
if ckpt_dir.exists():
    pth_files = sorted(ckpt_dir.glob("*.pth"))
    print(f"checkpoints/ ({len(pth_files)} file):")
    for f in pth_files:
        print(f"  {f.name:<25} {f.stat().st_size/1e6:>7.1f} MB")
runs_dir = working / "runs"
if runs_dir.exists():
    tb_files = list(runs_dir.rglob("events.out.tfevents.*"))
    print(f"runs/ ({len(tb_files)} event file, {sum(f.stat().st_size for f in tb_files)/1e6:.1f} MB)")